# SU(2) gate diffusion experiments

Scratch notebook for running package experiments against Colab GPU or a local VS Code/Jupyter kernel. Keep project logic in `su2diffusion`; use this notebook to choose configs, run reports, and compare outputs.

In [ ]:
# Run this in Colab or any fresh remote runtime before importing su2diffusion.
# Change BRANCH when testing a PR branch; use "main" after merge.
BRANCH = "main"

!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}

In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    center_names_for_config,
    diagnose_samples,
    get_experiment_config,
    plot_experiment_report,
    print_center_mass_table,
    print_conditional_label_table,
    print_diagnostics_table,
    print_per_center_table,
    resample_experiment,
    run_experiment,
)
from su2diffusion.data import centers_for_config

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
CONFIG_NAME = "baseline-clifford-cond"
EVAL_COUNT = 1000

config = get_experiment_config(CONFIG_NAME)
config = replace(config, sample_count=EVAL_COUNT, reference_count=EVAL_COUNT)
config

In [ ]:
result = run_experiment(config, device=device)

center_names = center_names_for_config(result.config.data)
print_diagnostics_table(result.diagnostics)
print()
print_center_mass_table(result.diagnostics, center_names=center_names)
if result.conditional_diagnostics is not None:
    print()
    print_conditional_label_table(result.conditional_diagnostics, center_names=center_names)
plot_experiment_report(result)

In [ ]:
# Eta sweep: reuse the trained model to tune within-gate spread without retraining.
ETA_SWEEP = [0.3, 0.5, 0.7, 1.0]
eta_results = resample_experiment(result, ETA_SWEEP, device=device)

print_diagnostics_table({name: item.diagnostics for name, item in eta_results.items()})
print()
print_center_mass_table({name: item.diagnostics for name, item in eta_results.items()}, center_names=center_names)
conditional_eta = {
    name: item.conditional_diagnostics
    for name, item in eta_results.items()
    if item.conditional_diagnostics is not None
}
if conditional_eta:
    print()
    print_conditional_label_table(conditional_eta, center_names=center_names)

In [ ]:
# Optional: slower, center-by-center gate diagnostics. Run only when the standard report looks interesting.
centers = centers_for_config(result.config.data, device=device)
deep_diagnostics = {
    name: diagnose_samples(
        samples,
        result.clean_reference,
        result.haar_reference,
        centers=centers,
        include_per_center=True,
    )
    for name, samples in {
        "deterministic": result.generated_deterministic,
        "stochastic": result.generated_stochastic,
    }.items()
}

print_diagnostics_table(deep_diagnostics)
print_per_center_table(deep_diagnostics, center_names=center_names)

In [ ]:
# Quick config comparison loop. Keep this to smoke/medium configs unless you want a long run.
for name in ["smoke", "smoke-gates", "smoke-gates-cond", "smoke-clifford-cond"]:
    cfg = get_experiment_config(name)
    cfg = replace(cfg, sample_count=512, reference_count=512)
    print("\n", cfg.name, cfg.data.kind, cfg.schedule.kind)
    quick = run_experiment(cfg, device=device)
    names = center_names_for_config(quick.config.data)
    print_diagnostics_table(quick.diagnostics)
    print_center_mass_table(quick.diagnostics, center_names=names)
    plot_experiment_report(quick)